# Train from Extracted Data (TRAIN_ONLY Mode) — Speculative Decoding Training

This notebook demonstrates how to train a custom **Eagle3 draft model** for speculative
decoding using the `SpeculativeDecodingTrainer` in `TRAIN_ONLY` mode on Red Hat OpenShift AI.

**TRAIN_ONLY** mode trains the Eagle3 draft model from **pre-extracted hidden states**.
No vLLM sidecar is deployed — the training container reads hidden state tensors directly
from the shared PVC. This is the second step of a two-step workflow that separates data
extraction from training, allowing you to iterate on hyperparameters without re-running
the expensive extraction step.

> **Prerequisite:** This notebook requires hidden states extracted by a completed
> [DATA_ONLY](../data-only/) run. Run the `data-only/` notebook first.

## What is Speculative Decoding?

Large language models generate tokens one at a time, and each token requires reading the
entire model from GPU memory — making inference **memory-bound**. Speculative decoding
exploits this: a small, fast **draft model** guesses the next several tokens, then the
large **verifier model** checks all guesses in a single forward pass. The output is
mathematically identical to normal decoding — no quality loss.

## What Gets Trained

Only the draft model's small components are trained — the verifier model is **frozen**
(never modified):

| Component | Purpose |
|-----------|--------|
| **FC layer 1 (fusion)** | Combines hidden states from four verifier layers into one vector |
| **FC layer 2 (concat)** | Merges the fused hidden state with the previous token's embedding |
| **One Transformer decoder layer** | Predicts the next token probability distribution |

## Hardware Requirements

| Component | GPU | CPU | Memory | Notes |
|-----------|-----|-----|--------|-------|
| Training container | 2× NVIDIA L40S / A100 | 4 cores | 64Gi | Runs Eagle3 draft model training |
| vLLM sidecar | Not needed | — | — | No extraction in this mode |

## Setup

Install the Kubeflow SDK and import required dependencies.

In [ ]:
# Install the Kubeflow SDK from the OpenDataHub fork (includes SpeculativeDecodingTrainer)
!pip install --no-cache-dir --force-reinstall --no-deps git+https://github.com/opendatahub-io/kubeflow-sdk.git@main

# Structured logging library (dependency for SDK progress tracking)
!pip install structlog

# Kubeflow Trainer API — provides TrainerClient, KubernetesBackendConfig, and job management
!pip install --no-cache-dir --force-reinstall --index-url https://pypi.org/simple kubeflow-trainer-api==2.3.0

In [ ]:
import os

import kubeflow

# Backend config tells the TrainerClient how to connect to the cluster
from kubeflow.common.types import KubernetesBackendConfig

# TrainerClient is the main entry point for submitting, monitoring, and deleting TrainJobs
from kubeflow.trainer import TrainerClient

# Name option lets you assign an explicit name to a TrainJob (otherwise auto-generated)
from kubeflow.trainer.options.common import Name

# Red Hat OpenShift AI extensions for speculative decoding training
from kubeflow.trainer.rhai import (
    SpeculativeDecodingTrainer,  # High-level trainer that wraps all four modes
    SpeculatorConfig,  # Fine-grained config: layer IDs, architecture, scheduler, etc.
    SpeculatorMode,  # Enum: DATA_ONLY, TRAIN_ONLY, OFFLINE, ONLINE
    SpeculatorType,  # Enum: EAGLE3 (currently the only supported type)
)

# Kubernetes Python client — used to configure API server auth and create the API client
from kubernetes import client

print(f"Kubeflow SDK version: {kubeflow.__version__}")
print("All imports successful")

In [ ]:
# Verify the SDK loaded correctly by inspecting the available enum values and defaults.
# This confirms that SpeculatorMode, SpeculatorType, and SpeculatorConfig are importable
# and behave as expected before proceeding to cluster authentication.
print(f"Modes: {[m.value for m in SpeculatorMode]}")
print(f"Types: {[t.value for t in SpeculatorType]}")
print(f"Config defaults: {SpeculatorConfig()}")
print("SDK ready")

## Authenticate to your OpenShift Cluster

Provide your OpenShift API server URL, authentication token, and HuggingFace token.
Update `PVC_NAME` to match the name of your shared RWX PersistentVolumeClaim.

In [ ]:
# ============================================================================
# CLUSTER AUTHENTICATION
# ============================================================================
# Replace these with your OpenShift cluster API server URL and bearer token.
# In OpenShift AI workbenches, these may be available as environment variables
# (OPENSHIFT_API_URL, NOTEBOOK_USER_TOKEN) — but for clarity we set them explicitly.
api_server = "<REPLACE WITH OPENSHIFT SERVER>"
token = "<REPLACE WITH OPENSHIFT TOKEN>"

# HuggingFace token — required for gated models; recommended for all models to avoid rate limits
HF_TOKEN = "<REPLACE WITH HF TOKEN>"

# ============================================================================
# KUBERNETES CLIENT CONFIGURATION
# ============================================================================
# The Configuration object holds the API server URL, auth token, and TLS settings.
# This is passed to the ApiClient, which the TrainerClient uses for all cluster operations.
configuration = client.Configuration()
configuration.host = api_server

# Uncomment if your cluster API server uses a self-signed certificate or an untrusted CA
# configuration.verify_ssl = False

configuration.api_key = {"authorization": f"Bearer {token}"}
api_client = client.ApiClient(configuration)

# ============================================================================
# PVC CONFIGURATION
# ============================================================================
# PVC_NAME must match the ReadWriteMany (RWX) PVC attached to your workbench.
# The notebook sees it at /opt/app-root/src/<pvc-name> (OpenShift AI convention).
# Training pods see it at /mnt/kubeflow-checkpoints (SDK constant CHECKPOINT_MOUNT_PATH).
PVC_NAME = "shared"
NOTEBOOK_SHARED_PATH = f"/opt/app-root/src/{PVC_NAME}"
SDK_MOUNT_PATH = "/mnt/kubeflow-checkpoints"

# Quick sanity check to help users discover the right workbench mount
if not os.path.exists(NOTEBOOK_SHARED_PATH):
    print(
        f"Warning: Expected workbench PVC mount not found at: {NOTEBOOK_SHARED_PATH}\n"
        "If your PVC has a different name or mount, update PVC_NAME above.\n"
        "Tip: in a workbench, PVCs are typically under /opt/app-root/src/."
    )

# ============================================================================
# TRAINER CLIENT AND CLUSTER TRAINING RUNTIME
# ============================================================================
# Create the TrainerClient — the main SDK entry point for submitting and managing TrainJobs.
trainer_client = TrainerClient(
    backend_config=KubernetesBackendConfig(
        client_configuration=api_client.configuration
    )
)

# TRAIN_ONLY mode uses the model optimization CTR (training only, no vLLM sidecar).
# Replace with the CTR name available on your cluster.
MODEL_OPT_CTR = "speculator-model-opt-cuda"

# Verify the CTR exists on the cluster
available_runtimes = {r.name for r in trainer_client.list_runtimes()}
status = (
    "Found" if MODEL_OPT_CTR in available_runtimes else "WARNING: not found on cluster"
)
print(f"CTR '{MODEL_OPT_CTR}': {status}")

print(f"\nAPI Server: {api_server}")
print(f"PVC name: {PVC_NAME}")
print(f"Workbench PVC mount: {NOTEBOOK_SHARED_PATH}")
print(f"Training pod PVC mount (SDK): {SDK_MOUNT_PATH}")

## Configuration

The following constants configure the training run. The verifier model is
[Qwen/Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B), a 36-layer transformer,
specified by its HuggingFace model ID. The training pods download the model
automatically — no manual pre-download is required (`HF_TOKEN` provides
authentication).

All output paths use **PVC URIs** (`pvc://<pvc-name>/<path>`), which the SDK
resolves to container mount paths internally.

`DATA_ONLY_OUTPUT` must point to the output directory from your completed
[DATA_ONLY](../data-only/) run. The TRAIN_ONLY trainer reads hidden state tensors
and the preprocessed dataset from this location.

`RUN_NAME` should match the run name used in the DATA_ONLY step.

In [ ]:
# Unique run identifier — must match the RUN_NAME used in the DATA_ONLY notebook
# so that hidden_states_path and data_path resolve to the correct DATA_ONLY output.
RUN_NAME = "run-01"

# Set the Verifier Model for the training job.
VERIFIER_MODEL = "Qwen/Qwen3-8B"

# Eagle3 reads hidden states from 4 intermediate layers of the verifier.
# Qwen3-8B has 36 transformer layers (indexed 1-36).
# Layers chosen: early (3), mid (18), late (33), and final (36) — giving the
# draft model a spread of low-level, mid-level, and high-level representations.
TARGET_LAYER_IDS = [3, 18, 33, 36]

# Output directory from the DATA_ONLY run — hidden states and preprocessed dataset
# are read from this path. This must match the output_dir used in the DATA_ONLY notebook.
DATA_ONLY_OUTPUT = f"pvc://{PVC_NAME}/speculator/{RUN_NAME}"

# GPU, CPU, and memory allocations for the training container.
# 2 GPUs enable data-parallel training; 64Gi memory holds model weights + optimizer state.
TRAINING_RESOURCES = {
    "nvidia.com/gpu": 2,
    "cpu": "4",
    "memory": "64Gi",
}

# Training hyperparameters
EPOCHS = 3  # Number of full passes over the training data
LEARNING_RATE = 1e-4  # AdamW learning rate — 1e-4 is a good starting point for Eagle3
TOTAL_SEQ_LEN = 2048  # Maximum sequence length for training

print("Configuration:")
print(f"  Run name:          {RUN_NAME}")
print(f"  Verifier model:    {VERIFIER_MODEL}")
print(f"  Target layers:     {TARGET_LAYER_IDS}")
print(f"  DATA_ONLY output:  {DATA_ONLY_OUTPUT}")
print(f"  Training GPUs:     {TRAINING_RESOURCES['nvidia.com/gpu']}")
print(f"  Epochs:            {EPOCHS}")
print(f"  Learning rate:     {LEARNING_RATE}")
print(f"  Sequence length:   {TOTAL_SEQ_LEN}")

## Train from Extracted Data (TRAIN_ONLY Mode)

The `TRAIN_ONLY` mode trains the Eagle3 draft model using hidden states extracted in
a previous `DATA_ONLY` run. No vLLM sidecar is needed — the training container reads
directly from the PVC.

Only the draft model's small components are trained:
- **FC layer 1 (fusion)**: Combines hidden states from four verifier layers into one vector
- **FC layer 2 (concat)**: Merges the fused hidden state with the previous token's embedding
- **One Transformer decoder layer**: Predicts the next token probability distribution

The verifier model is frozen — never modified.

**Key parameters explained:**
- `hidden_states_path` — Points to the `hidden_states/` subdirectory created by `DATA_ONLY`
- `data_path` — Points to the `DATA_ONLY` output directory (contains the preprocessed dataset)
- `num_layers` — Number of Transformer decoder layers in the draft model (1 is default)
- `ttt_steps` — Test-time training steps per batch
- `norm_before_residual` — Apply layer normalization before the residual connection
- `scheduler_type` — Learning rate scheduler (`"linear"` decays LR linearly to zero)
- `checkpoint_freq` — Save a checkpoint every N epochs (1.0 = every epoch)
- `resume_from_checkpoint` — If `True`, resumes training from the latest checkpoint if one exists

In [ ]:
TRAIN_JOB = f"eagle3-train-{RUN_NAME}"
TRAIN_ONLY_OUTPUT = f"pvc://{PVC_NAME}/speculator/{RUN_NAME}/checkpoints"

# Configure the TRAIN_ONLY trainer.
# This mode trains the Eagle3 draft model from pre-extracted hidden states.
# No vLLM sidecar is deployed — the training container reads directly from PVC.
train_only_trainer = SpeculativeDecodingTrainer(
    mode=SpeculatorMode.TRAIN_ONLY,  # Train from existing hidden states
    speculator_type=SpeculatorType.EAGLE3,
    verifier_model=VERIFIER_MODEL,
    hidden_states_path=f"{DATA_ONLY_OUTPUT}/hidden_states",  # Points to DATA_ONLY output
    data_path=DATA_ONLY_OUTPUT,  # Preprocessed dataset from DATA_ONLY
    training_resources=TRAINING_RESOURCES,  # 2 GPUs for data-parallel training
    enable_progression_tracking=True,  # Enable SDK-side progress polling
    packages_to_install=["speculators==0.6.0", "torchvision==0.24.0"],
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    total_seq_len=TOTAL_SEQ_LEN,
    output_dir=TRAIN_ONLY_OUTPUT,
    config=SpeculatorConfig(
        target_layer_ids=TARGET_LAYER_IDS,
        num_layers=1,  # Number of transformer decoder layers in the draft model
        ttt_steps=3,  # Test-time training steps per batch (refine on each batch)
        norm_before_residual=True,  # Apply LayerNorm before adding the residual connection
        scheduler_type="linear",  # Linearly decay learning rate to zero over training
        checkpoint_freq=1.0,  # Save a checkpoint after every epoch
        resume_from_checkpoint=True,  # Resume from the latest checkpoint if one exists
    ),
    env={"HF_TOKEN": HF_TOKEN},
)

print("TRAIN_ONLY Configuration:")
print(f"  Job name:           {TRAIN_JOB}")
print(f"  Mode:               {train_only_trainer.mode.value}")
print(f"  Verifier:           {train_only_trainer.verifier_model}")
print(f"  Hidden states path: {train_only_trainer.hidden_states_path}")
print(f"  Data path:          {train_only_trainer.data_path}")
print(f"  Target layers:      {train_only_trainer.config.target_layer_ids}")
print(f"  Epochs:             {train_only_trainer.epochs}")
print(f"  Learning rate:      {train_only_trainer.lr}")
print(f"  Output dir:         {train_only_trainer.output_dir}")

In [ ]:
# Submit the TRAIN_ONLY TrainJob to the cluster.
# Uses MODEL_OPT_CTR since no vLLM sidecar is needed — training only.
trainer_client.train(
    options=[Name(name=TRAIN_JOB)],
    trainer=train_only_trainer,
    runtime=MODEL_OPT_CTR,
)

print(f"TRAIN_ONLY job submitted: {TRAIN_JOB}")
print("\nMonitor logs with:")
print(f"  oc logs -f -l batch.kubernetes.io/job-name={TRAIN_JOB}-node-0 -c node")

In [ ]:
# Check the current status of the TRAIN_ONLY job.
# Re-run this cell periodically to poll for completion.
trainer_client.get_job(TRAIN_JOB)

## Cleanup

Delete the TrainJob when you are done. Uncomment the line below to delete.

In [ ]:
# Delete the completed TrainJob to free cluster resources (pods, volumes, etc.).
# Note: Deleting a job does NOT delete the output data on the PVC —
# checkpoints remain available for future use.

# trainer_client.delete_job(TRAIN_JOB)
# print("TrainJob deleted.")

## Summary

This notebook trained an Eagle3 draft model from pre-extracted hidden states using
the `TRAIN_ONLY` mode of `SpeculativeDecodingTrainer`.

### Key Takeaways

- The **DATA_ONLY + TRAIN_ONLY** split lets you extract data once and iterate on
  training hyperparameters (`epochs`, `lr`, `num_layers`, `ttt_steps`, `scheduler_type`)
  without re-running the expensive extraction step.
- Use `resume_from_checkpoint=True` to resume training after interruptions.
- The trained draft model checkpoints are saved to the PVC at the configured `output_dir`.

### Next Steps

- **Deploy** the trained draft model with vLLM for speculative decoding inference.
- **Adjust hyperparameters** — change `epochs`, `lr`, `num_layers`, or `ttt_steps`
  and re-run this notebook without re-extracting hidden states.
- **Try [ONLINE](../online/) mode** for a simpler single-step alternative that
  combines extraction and training in one job.